# 常用分块方法

分块没有一个通用答案。本页介绍定长、按分隔符、递归分隔、按文档格式、按模型输入长度和按语义变化分块。它们解决的问题不同，因此不提供一份适用于所有资料的排名。

## 定长、按分隔符和递归字符分块

定长分块简单且大小稳定，但可能从句子中间切开。按分隔符分块会尽量保留完整段落。递归分隔会先尝试段落，过长时再尝试句号、逗号，必要时按长度硬切。

In [1]:
import re

text = "模型评估要区分训练误差和测试误差。类别不平衡时，还要区分宏平均和微平均。"

def fixed_split(value, size):
    return [value[start:start + size] for start in range(0, len(value), size)]

def separator_split(value, separator="。"):
    return [part + separator for part in value.split(separator) if part]

def recursive_split(value, size):
    result = []
    for sentence in separator_split(value):
        if len(sentence) <= size:
            result.append(sentence)
        else:
            result.extend(fixed_split(sentence, size))
    return result

print("定长分块：", fixed_split(text, 18))
print("按句号分块：", separator_split(text))
print("递归字符分块：", recursive_split(text, 32))

定长分块： ['模型评估要区分训练误差和测试误差。类', '别不平衡时，还要区分宏平均和微平均。']
按句号分块： ['模型评估要区分训练误差和测试误差。', '类别不平衡时，还要区分宏平均和微平均。']
递归字符分块： ['模型评估要区分训练误差和测试误差。', '类别不平衡时，还要区分宏平均和微平均。']


## 按文档格式分块

Markdown、HTML 和代码本身已经有标题、标签或函数边界。先使用这些边界，可以避免把标题和它所属的正文分开。

In [2]:
markdown_text = "# 安装\n先创建环境。\n## 运行\n再启动 Notebook。\n## 排查\n先看报错信息。"

sections = [part.strip() for part in re.split(r"(?=^#{1,6} )", markdown_text, flags=re.M) if part.strip()]
print("按 Markdown 标题分块：")
for section in sections:
    print(repr(section))
assert all(section.startswith("#") for section in sections)

按 Markdown 标题分块：
'# 安装\n先创建环境。'
'## 运行\n再启动 Notebook。'
'## 排查\n先看报错信息。'


## 按模型输入长度分块

有些向量模型只读取有限长度。这时应先使用该模型的分词器计数，再按上限分块。字符数不等于 token 数，下面只用已经分好的 token 列表演示“不超过上限”的逻辑。

In [3]:
tokens = ["模型", "评估", "需要", "独立", "测试集", "和", "稳定", "指标"]
limit = 3
token_chunks = [tokens[start:start + limit] for start in range(0, len(tokens), limit)]
print("token 上限：", limit)
print("分块结果：", token_chunks)
assert max(map(len, token_chunks)) <= limit

token 上限： 3
分块结果： [['模型', '评估', '需要'], ['独立', '测试集', '和'], ['稳定', '指标']]


## 按语义变化分块

语义分块会比较相邻句子。主题明显改变时开始新片段，因此长度不固定。它适合主题边界比段落边界更重要的资料，但建立索引的成本更高，也必须用项目问题测试。下面用字符二元组做可见的小示例；实际项目通常使用向量模型计算相似度。

In [4]:
sentences = [
    "模型评估比较模型的好坏。",
    "模型评估还需要独立测试集。",
    "数据库保存订单和用户记录。",
    "数据库访问需要权限控制。",
]

def bigrams(value):
    chars = [char for char in value if not char.isspace()]
    return {"".join(chars[index:index + 2]) for index in range(len(chars) - 1)}

def similarity(left, right):
    a, b = bigrams(left), bigrams(right)
    return len(a & b) / max(len(a | b), 1)

semantic_chunks = [[sentences[0]]]
for previous, current in zip(sentences, sentences[1:]):
    if similarity(previous, current) < 0.05:
        semantic_chunks.append([])
    semantic_chunks[-1].append(current)

print("相邻句子相似度：", [round(similarity(a, b), 3) for a, b in zip(sentences, sentences[1:])])
print("按语义变化分成：", ["".join(group) for group in semantic_chunks])
assert len(semantic_chunks) == 2

相邻句子相似度： [0.158, 0.0, 0.095]
按语义变化分成： ['模型评估比较模型的好坏。模型评估还需要独立测试集。', '数据库保存订单和用户记录。数据库访问需要权限控制。']


## 怎样选

| 资料特点 | 先尝试 |
|---|---|
| 没有结构，只需要稳定长度 | 定长分块 |
| 句子或段落边界可靠 | 按分隔符或递归分隔 |
| Markdown、HTML、代码 | 先按文档格式分 |
| 接近模型输入上限 | 用模型的分词器计数 |
| 段落中频繁切换主题 | 试验语义分块 |

这些方法的详细效果不应靠方法名推测。下一页会用固定问题比较分块设置；[分块优化](分块优化.ipynb)则说明按页和按句子边界的完整对照。

## 为什么分块会改变检索结果

分块要兼顾完整和简短。单个片段需要保住一个可以回答的意思，片段又不能无限变长，因为向量模型有输入上限，回答模型也要阅读所有找到的文字。token 和字符不是同一单位，中文、英文、数字和标点的切分方式不同；实际项目应使用目标模型的 tokenizer 计数，不能把字符数直接当 token 数。

定长分块最稳定，公式可以写成 start = i * (chunk_size - chunk_overlap)、end = start + chunk_size。chunk_overlap 能降低边界丢信息的概率，但会增加片段数和重复上下文；重叠不能替代合理的边界。按分隔符分块会优先保留段落或句子，不过某个段落本身过长时仍可能超过上限。CharacterTextSplitter 找不到合适分隔符时会给出超长片段警告，这不是可以忽略的错误，而是需要调整分隔符或二次切分的信号。

RecursiveCharacterTextSplitter 按分隔符优先级递归尝试，例如先尝试两个换行、一个换行、空格，必要时按任意字符切开。它通常比单一分隔符更能保住结构，但中文资料要显式加入句号、逗号等分隔符，并检查是否切断公式、列表和代码。


## 文档格式、token 上限和语义分块

Markdown、HTML 和代码已经带有标题、标签或函数边界。MarkdownTextSplitter、HTMLHeaderTextSplitter 等方法先利用这些边界，再处理过长的正文，可以避免把标题和正文完全拆开。标题不一定要独立成为一个片段；很多任务更适合把标题和下面一段正文放在一起。

按 token 分块时要使用与向量模型相同或兼容的 tokenizer。token 经过编码再解码后，可能出现空格变化或 [UNK]（词表没有收录的标记），所以要在目标语言和目标模型上抽查内容，不能只看长度是否满足上限。

语义分块先按句号、问号和感叹号得到句子，再把相邻句子组成窗口并生成向量，计算相邻窗口的距离；距离突然变大处可以作为主题边界。阈值可以按百分位数、标准差或四分位距计算。它能处理段落边界不可靠的资料，但片段长度不固定，建立索引也更慢；如果片段超过模型输入上限，还要再次切分。下面的示意图展示相邻句子距离和切分位置的关系。

![语义分块的相邻句子示意](figures/Semantic_Chunker_1.png)

没有一种分块方法天然最好：说明性 PDF 可能适合按页，结构化 Markdown 适合按标题，句子规则连续的数学文字可能更适合按句子。最终要在自己的问题集上比较必要页面命中、证据是否完整、片段数量和上下文字符数。


## 其他分块器的代码写法

下面只展示常见分块器的代码写法，不产生本页的比较结果；这里的 embedding 指把文字转换成向量的模型。运行前需安装对应依赖，并重新用本页的问题集核对片段边界。

```python
# API 使用示例；运行前请安装对应的 langchain-text-splitters
from langchain_text_splitters import (
    CharacterTextSplitter,
    MarkdownTextSplitter,
    RecursiveCharacterTextSplitter,
)
from langchain_text_splitters.sentence_transformers import (
    SentenceTransformersTokenTextSplitter,
)

sample = "第一段说明。第二段说明包含更长的解释，需要在句子边界处理。"
fixed = CharacterTextSplitter(
    chunk_size=30, chunk_overlap=5, separator="", keep_separator=False
).split_text(sample)
recursive = RecursiveCharacterTextSplitter(
    chunk_size=30,
    chunk_overlap=5,
    separators=["\n\n", "\n", "。", "，", " ", ""],
).split_text(sample)
markdown_chunks = MarkdownTextSplitter(
    chunk_size=300, chunk_overlap=50, keep_separator=False
).split_text("# 安装\n先创建环境。\n## 运行\n再启动 Notebook。")

# token_splitter.count_tokens(text=chunk) 应使用与向量表示模型一致的 tokenizer。
# SemanticChunker 需要向量表示模型，且建议再检查每个结果是否超过 max_tokens。
print("定长：", fixed)
print("递归：", recursive)
print("Markdown：", markdown_chunks)
```
